# Retail Sales Forecasting and Holiday Analysis

This notebook continues the retail sales data preparation and EDA pipeline in `01_data_preparation_and_eda.ipynb`.

It includes histogram gradient boosting, a seasonal-naive benchmark, chronological evaluation, markdown scenario comparisons and holiday analysis. Its original development context was an academic project; this is a portfolio-formatted copy. AI assistance with the benchmark and analysis design is documented in the project methodology notes.

**Prerequisite:** Run the EDA notebook after downloading the Kaggle Sales, Features and Stores CSVs, so that `02_Cleaned_Data/retail_analysis_data.csv` exists.


## 1. Setup and Prepared Dataset


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

# Find the prepared Task 1 dataset in common project locations.
current_folder = Path.cwd()

candidate_files = [
    current_folder / "retail_analysis_data.csv",
    current_folder / "retail_analysis_data(1).csv",
    current_folder / "02_Cleaned_Data" / "retail_analysis_data.csv",
    current_folder.parent / "retail_analysis_data.csv",
    current_folder.parent / "retail_analysis_data(1).csv",
    current_folder.parent / "02_Cleaned_Data" / "retail_analysis_data.csv",
]

data_file = next((path for path in candidate_files if path.exists()), None)

if data_file is None:
    raise FileNotFoundError(
        "retail_analysis_data.csv was not found. Place the cleaned Task 1 file "
        "beside this notebook or in the 02_Cleaned_Data folder."
    )

output_folder = current_folder / "task2_outputs"
chart_folder = current_folder / "task2_charts"
output_folder.mkdir(exist_ok=True)
chart_folder.mkdir(exist_ok=True)

print("Prepared dataset:", data_file)
print("Output folder:", output_folder)
print("Chart folder:", chart_folder)


In [ ]:
analysis_data = pd.read_csv(
    data_file,
    parse_dates=["Date"]
)

analysis_data = analysis_data.sort_values(
    ["Store", "Dept", "Date"]
).reset_index(drop=True)

print("Rows:", f"{len(analysis_data):,}")
print("Columns:", analysis_data.shape[1])
print("Stores:", analysis_data["Store"].nunique())
print("Departments:", analysis_data["Dept"].nunique())
print(
    "Historical date range:",
    analysis_data["Date"].min().date(),
    "to",
    analysis_data["Date"].max().date()
)

display(analysis_data.head())


### 1.1 Modelling approach

The train/test split is **chronological**, not random, because random splitting would mix future information into the training sample. The final 12 observed weeks are held out for evaluation.

the primary model is a **Histogram Gradient Boosting regression model using time-series lag variables**. The model uses store/department identifiers, store characteristics, calendar variables, holiday status, markdown values and historical sales lags. Lag and rolling variables are shifted so the current week's target is never used as an input.

The AI-assisted benchmark is a **52-week seasonal-naive forecast**: the predicted sales for a store–department series equal the sales recorded in the same seasonal week one year earlier. This benchmark was selected as an AI-assisted comparison because retail sales show strong annual seasonality and the result can be independently reproduced and checked in Python.


## 2. Time-Series Feature Engineering


In [ ]:
model_frame = analysis_data.copy()

# Convert text/Boolean fields to numerical modelling fields.
model_frame["Type_Code"] = model_frame["Type"].map({"A": 0, "B": 1, "C": 2})
model_frame["IsHoliday_Int"] = model_frame["IsHoliday"].astype(int)
model_frame["Total_MarkDown_Filled"] = model_frame["Total_MarkDown"].fillna(0)
model_frame["Markdown_Reported_Int"] = model_frame["Markdown_Reported"].astype(int)

series_group = model_frame.groupby(["Store", "Dept"], sort=False)

# Historical sales lags. Shift ensures only earlier sales are used.
lag_settings = {
    1: 7,
    4: 28,
    13: 91,
    52: 364,
}

exact_lag_check = pd.Series(True, index=model_frame.index)

for lag_weeks, expected_days in lag_settings.items():
    model_frame[f"Lag_{lag_weeks}"] = (
        series_group["Weekly_Sales"].shift(lag_weeks)
    )
    shifted_date = series_group["Date"].shift(lag_weeks)
    exact_lag_check &= (
        (model_frame["Date"] - shifted_date).dt.days == expected_days
    )

# Rolling means are also shifted by one week to prevent target leakage.
model_frame["Rolling_4"] = (
    series_group["Weekly_Sales"]
    .transform(lambda s: s.shift(1).rolling(4, min_periods=4).mean())
)

model_frame["Rolling_13"] = (
    series_group["Weekly_Sales"]
    .transform(lambda s: s.shift(1).rolling(13, min_periods=13).mean())
)

feature_columns = [
    "Store",
    "Dept",
    "Type_Code",
    "Size",
    "Month_Number",
    "Week_Number",
    "Quarter",
    "IsHoliday_Int",
    "Total_MarkDown_Filled",
    "Markdown_Reported_Int",
    "Lag_1",
    "Lag_4",
    "Lag_13",
    "Lag_52",
    "Rolling_4",
    "Rolling_13",
]

# Evaluation uses only rows where the calendar lags line up exactly.
model_data = model_frame.loc[exact_lag_check].dropna(
    subset=["Rolling_4", "Rolling_13"]
).copy()

print("Rows available for modelling:", f"{len(model_data):,}")
print(
    "Model date range:",
    model_data["Date"].min().date(),
    "to",
    model_data["Date"].max().date()
)


## 3. Sales Forecasting Model and Evaluation


In [ ]:
# Hold out the final 12 historical weeks for a realistic forecasting test.
latest_date = model_data["Date"].max()
test_start = latest_date - pd.Timedelta(weeks=11)
train_end = test_start - pd.Timedelta(days=7)

train_data = model_data[model_data["Date"] <= train_end].copy()
test_data = model_data[model_data["Date"] >= test_start].copy()

print("Training ends:", train_end.date())
print("Test period:", test_data["Date"].min().date(), "to", test_data["Date"].max().date())
print("Training rows:", f"{len(train_data):,}")
print("Test rows:", f"{len(test_data):,}")
print("Store-department series in test:", test_data[["Store", "Dept"]].drop_duplicates().shape[0])


In [ ]:
# The model settings balance accuracy, runtime and interpretability for demonstration.
forecast_model = HistGradientBoostingRegressor(
    max_iter=180,
    learning_rate=0.08,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

X_train = train_data[feature_columns]
y_train = train_data["Weekly_Sales"]
X_test = test_data[feature_columns]
y_test = test_data["Weekly_Sales"]

forecast_model.fit(X_train, y_train)

own_predictions = forecast_model.predict(X_test)

# AI-assisted comparison: same store/department sales from 52 weeks earlier.
ai_predictions = test_data["Lag_52"].to_numpy()

print("Forecasting model fitted and test predictions created.")


In [ ]:
def regression_metrics(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    wape = (
        np.abs(actual - predicted).sum()
        / np.abs(actual).sum()
    ) * 100

    # This is a forecasting accuracy measure derived from WAPE,
    # not classification accuracy.
    forecast_accuracy = 100 - wape
    r2 = r2_score(actual, predicted)

    return {
        "RMSE": rmse,
        "MAE": mae,
        "WAPE_Percent": wape,
        "Forecast_Accuracy_Percent": forecast_accuracy,
        "R2": r2,
    }

own_metrics = regression_metrics(y_test, own_predictions)
ai_metrics = regression_metrics(y_test, ai_predictions)

model_metrics = pd.DataFrame([
    {"Model": "Own HistGradientBoosting lag model", **own_metrics},
    {"Model": "AI-assisted 52-week seasonal naive", **ai_metrics},
])

model_metrics.to_csv(
    output_folder / "task2_sales_forecasting_metrics.csv",
    index=False
)

display(model_metrics.round(3))


**How to read the metrics:** lower RMSE, MAE and WAPE are better. The WAPE-based forecast accuracy is shown because the original assessment guide gives accuracy as an example metric; it is calculated as `100 − WAPE`. R² is also reported to show how much variation is captured by the forecasts.


In [ ]:
# Store each row-level result so it can be checked or reused by the group.
test_results = test_data[
    ["Date", "Store", "Dept", "Weekly_Sales", "IsHoliday", "Holiday_Name"]
].copy()

test_results["Own_Model_Prediction"] = own_predictions
test_results["AI_Seasonal_Naive_Prediction"] = ai_predictions

test_results.to_csv(
    output_folder / "task2_test_predictions.csv",
    index=False
)

# Calculate metrics separately for each evaluated store-department series.
def series_metric_row(group):
    actual = group["Weekly_Sales"].to_numpy()
    own = group["Own_Model_Prediction"].to_numpy()
    ai = group["AI_Seasonal_Naive_Prediction"].to_numpy()

    own_wape = np.abs(actual - own).sum() / np.abs(actual).sum() * 100
    ai_wape = np.abs(actual - ai).sum() / np.abs(actual).sum() * 100

    return pd.Series({
        "Weeks_Evaluated": len(group),
        "Own_RMSE": np.sqrt(mean_squared_error(actual, own)),
        "AI_RMSE": np.sqrt(mean_squared_error(actual, ai)),
        "Own_WAPE_Percent": own_wape,
        "AI_WAPE_Percent": ai_wape,
    })

series_metric_rows = []

for (store, dept), group in test_results.groupby(["Store", "Dept"]):
    metric_values = series_metric_row(group).to_dict()
    metric_values["Store"] = store
    metric_values["Dept"] = dept
    series_metric_rows.append(metric_values)

store_department_metrics = pd.DataFrame(series_metric_rows)[
    [
        "Store", "Dept", "Weeks_Evaluated",
        "Own_RMSE", "AI_RMSE",
        "Own_WAPE_Percent", "AI_WAPE_Percent"
    ]
]

store_department_metrics.to_csv(
    output_folder / "task2_store_department_metrics.csv",
    index=False
)

print("Per-series metric rows:", len(store_department_metrics))


In [ ]:
# Aggregate the test results by week for a clear visual comparison.
weekly_test = (
    test_results
    .groupby("Date", as_index=False)
    .agg(
        Actual_Sales=("Weekly_Sales", "sum"),
        Own_Model=("Own_Model_Prediction", "sum"),
        AI_Benchmark=("AI_Seasonal_Naive_Prediction", "sum"),
    )
)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(weekly_test["Date"], weekly_test["Actual_Sales"] / 1_000_000, marker="o", label="Actual")
ax.plot(weekly_test["Date"], weekly_test["Own_Model"] / 1_000_000, marker="o", label="Own model")
ax.plot(weekly_test["Date"], weekly_test["AI_Benchmark"] / 1_000_000, marker="o", label="AI benchmark")
ax.set_title("12-Week Forecast Evaluation: Actual vs Predicted Sales")
ax.set_xlabel("Week")
ax.set_ylabel("Total Sales ($ Millions)")
ax.grid(alpha=0.25)
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()

figure_1 = chart_folder / "Figure_8_Task2_Forecast_Evaluation.png"
plt.savefig(figure_1, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", figure_1)


### 3.1 Feature importance


In [ ]:
# Permutation importance measures how much prediction quality drops when
# one feature is shuffled. A sample is used to keep this check fast.
importance_sample = test_data.sample(
    n=min(3000, len(test_data)),
    random_state=42
)

importance_result = permutation_importance(
    forecast_model,
    importance_sample[feature_columns],
    importance_sample["Weekly_Sales"],
    n_repeats=3,
    random_state=42,
    scoring="neg_mean_absolute_error"
)

feature_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": importance_result.importances_mean
}).sort_values("Importance", ascending=False)

feature_importance.to_csv(
    output_folder / "task2_feature_importance.csv",
    index=False
)

display(feature_importance.head(10).round(4))


## 4. Markdown Impact Scenario


The assessment objective also refers to markdown events. To examine their possible effect without claiming causation, the fitted forecasting model is used for a **scenario comparison** on the test period:

- Scenario A uses the recorded markdown amount.
- Scenario B keeps the same store, department, date and historical sales inputs but sets the markdown amount to zero.

The difference is a **model-estimated association**, not proof that markdown caused the sales change. Other promotional or business factors may occur at the same time.


In [ ]:
X_no_markdown = X_test.copy()
X_no_markdown["Total_MarkDown_Filled"] = 0

no_markdown_predictions = forecast_model.predict(X_no_markdown)
markdown_effect = own_predictions - no_markdown_predictions

markdown_impact = test_data[
    ["Date", "Store", "Dept", "Type", "Total_MarkDown"]
].copy()

markdown_impact["Predicted_With_Recorded_MarkDown"] = own_predictions
markdown_impact["Predicted_With_Zero_MarkDown"] = no_markdown_predictions
markdown_impact["Estimated_MarkDown_Impact"] = markdown_effect

markdown_impact.to_csv(
    output_folder / "task2_markdown_impact_rows.csv",
    index=False
)

markdown_summary = (
    markdown_impact
    .groupby("Type", as_index=False)
    .agg(
        Records=("Estimated_MarkDown_Impact", "size"),
        Mean_Impact_Per_Record=("Estimated_MarkDown_Impact", "mean"),
        Median_Impact_Per_Record=("Estimated_MarkDown_Impact", "median"),
        Total_Modelled_Impact=("Estimated_MarkDown_Impact", "sum"),
    )
)

overall_row = pd.DataFrame({
    "Type": ["All"],
    "Records": [len(markdown_impact)],
    "Mean_Impact_Per_Record": [markdown_impact["Estimated_MarkDown_Impact"].mean()],
    "Median_Impact_Per_Record": [markdown_impact["Estimated_MarkDown_Impact"].median()],
    "Total_Modelled_Impact": [markdown_impact["Estimated_MarkDown_Impact"].sum()],
})

markdown_summary = pd.concat([overall_row, markdown_summary], ignore_index=True)

markdown_summary.to_csv(
    output_folder / "task2_markdown_impact_summary.csv",
    index=False
)

display(markdown_summary.round(2))


In [ ]:
plot_markdown = markdown_summary[markdown_summary["Type"] != "All"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    plot_markdown["Type"],
    plot_markdown["Mean_Impact_Per_Record"]
)
ax.bar_label(bars, fmt="$%.1f", padding=3)
ax.set_title("Model-Estimated Markdown Impact by Store Type")
ax.set_xlabel("Store Type")
ax.set_ylabel("Mean Predicted Sales Difference per Department-Week ($)")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()

figure_2 = chart_folder / "Figure_9_Task2_Markdown_Impact.png"
plt.savefig(figure_2, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", figure_2)


## 5. Future Sales Forecast for Active Store–Department Series


The final own model is refitted using all available historical modelling rows and then used to forecast the next **nine weekly periods**, from 2 November 2012 through 28 December 2012. This horizon includes the labelled Thanksgiving and Christmas weeks.

The future baseline assumes **no planned markdown amount is supplied**. If future data provides future markdown plans, the `Total_MarkDown_Filled` input can be replaced and the forecast rerun.

All store–department series active on the last historical date are included. Where a newly active or irregular series does not have an exact 52-week lag, the recent 13-week mean is used as a fallback for missing lag inputs and the AI benchmark. The output flags these fallback cases.


In [ ]:
final_model = HistGradientBoostingRegressor(
    max_iter=180,
    learning_rate=0.08,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

final_model.fit(
    model_data[feature_columns],
    model_data["Weekly_Sales"]
)

historical_end = analysis_data["Date"].max()
active_series = (
    analysis_data[analysis_data["Date"] == historical_end]
    [["Store", "Dept", "Type", "Size"]]
    .drop_duplicates()
    .copy()
)
active_series["Type_Code"] = active_series["Type"].map({"A": 0, "B": 1, "C": 2})

print("Active store-department series:", len(active_series))
print("Stores represented:", active_series["Store"].nunique())
print("Departments represented:", active_series["Dept"].nunique())


In [ ]:
# Store each series as a date -> sales dictionary so future predictions can be
# fed back into later lag values during the recursive forecast.
history = {
    (int(store), int(dept)): dict(zip(group["Date"], group["Weekly_Sales"]))
    for (store, dept), group in analysis_data.groupby(["Store", "Dept"])
}

future_dates = pd.date_range(
    historical_end + pd.Timedelta(days=7),
    periods=9,
    freq="7D"
)

future_holiday_map = {
    pd.Timestamp("2012-11-23"): "Thanksgiving",
    pd.Timestamp("2012-12-28"): "Christmas",
}

future_rows = []

for forecast_date in future_dates:
    feature_rows = []
    metadata_rows = []

    for row in active_series.itertuples(index=False):
        key = (int(row.Store), int(row.Dept))
        series_history = history[key]

        # Exact weekly values where available.
        def exact_value(weeks_back):
            target_date = forecast_date - pd.Timedelta(weeks=weeks_back)
            return series_history.get(target_date, np.nan)

        previous_13 = [exact_value(i) for i in range(1, 14)]
        available_13 = [value for value in previous_13 if pd.notna(value)]

        if available_13:
            recent_13_mean = float(np.mean(available_13))
        else:
            # Very conservative fallback; this should be rare for an active series.
            recent_values = list(series_history.values())[-13:]
            recent_13_mean = float(np.mean(recent_values))

        previous_4 = [exact_value(i) for i in range(1, 5)]
        available_4 = [value for value in previous_4 if pd.notna(value)]
        recent_4_mean = (
            float(np.mean(available_4))
            if available_4
            else recent_13_mean
        )

        lag_1 = exact_value(1)
        lag_4 = exact_value(4)
        lag_13 = exact_value(13)
        lag_52 = exact_value(52)

        lag_1_fallback = pd.isna(lag_1)
        lag_4_fallback = pd.isna(lag_4)
        lag_13_fallback = pd.isna(lag_13)
        lag_52_fallback = pd.isna(lag_52)

        if lag_1_fallback:
            lag_1 = recent_4_mean
        if lag_4_fallback:
            lag_4 = recent_4_mean
        if lag_13_fallback:
            lag_13 = recent_13_mean
        if lag_52_fallback:
            lag_52 = recent_13_mean

        is_holiday = int(forecast_date in future_holiday_map)

        feature_rows.append({
            "Store": row.Store,
            "Dept": row.Dept,
            "Type_Code": row.Type_Code,
            "Size": row.Size,
            "Month_Number": forecast_date.month,
            "Week_Number": int(forecast_date.isocalendar().week),
            "Quarter": forecast_date.quarter,
            "IsHoliday_Int": is_holiday,
            "Total_MarkDown_Filled": 0.0,
            "Markdown_Reported_Int": 0,
            "Lag_1": lag_1,
            "Lag_4": lag_4,
            "Lag_13": lag_13,
            "Lag_52": lag_52,
            "Rolling_4": recent_4_mean,
            "Rolling_13": recent_13_mean,
        })

        metadata_rows.append({
            "Date": forecast_date,
            "Store": int(row.Store),
            "Dept": int(row.Dept),
            "Type": row.Type,
            "Size": int(row.Size),
            "IsHoliday": bool(is_holiday),
            "Holiday_Name": future_holiday_map.get(forecast_date, "Non-holiday"),
            "AI_52Week_Fallback_Used": bool(lag_52_fallback),
            "AI_Seasonal_Naive_Forecast": float(lag_52),
        })

    future_X = pd.DataFrame(feature_rows)[feature_columns]
    own_future = final_model.predict(future_X)

    for meta, prediction in zip(metadata_rows, own_future):
        meta["Own_Model_Forecast"] = float(prediction)
        future_rows.append(meta)

        # Feed the own-model forecast into the own-model history for later lags.
        key = (meta["Store"], meta["Dept"])
        history[key][forecast_date] = float(prediction)

future_forecast = pd.DataFrame(future_rows)

future_forecast = future_forecast[
    [
        "Date", "Store", "Dept", "Type", "Size", "IsHoliday", "Holiday_Name",
        "Own_Model_Forecast", "AI_Seasonal_Naive_Forecast",
        "AI_52Week_Fallback_Used"
    ]
]

future_forecast.to_csv(
    output_folder / "task2_future_store_department_forecast.csv",
    index=False
)

print("Future forecast rows:", f"{len(future_forecast):,}")
print("Forecast weeks:", future_forecast["Date"].nunique())
print("Active series per week:", future_forecast.groupby("Date").size().min(), "to", future_forecast.groupby("Date").size().max())
print("AI lag-52 fallback rows:", future_forecast["AI_52Week_Fallback_Used"].sum())


In [ ]:
future_weekly = (
    future_forecast
    .groupby(["Date", "Holiday_Name"], as_index=False)
    .agg(
        Own_Model_Forecast=("Own_Model_Forecast", "sum"),
        AI_Benchmark_Forecast=("AI_Seasonal_Naive_Forecast", "sum")
    )
)

future_weekly.to_csv(
    output_folder / "task2_future_weekly_summary.csv",
    index=False
)

display(future_weekly)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(future_weekly["Date"], future_weekly["Own_Model_Forecast"] / 1_000_000, marker="o", label="Own model")
ax.plot(future_weekly["Date"], future_weekly["AI_Benchmark_Forecast"] / 1_000_000, marker="o", label="AI benchmark")
ax.set_title("Nine-Week Future Sales Forecast")
ax.set_xlabel("Forecast Week")
ax.set_ylabel("Total Forecast Sales ($ Millions)")
ax.grid(alpha=0.25)
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()

figure_3 = chart_folder / "Figure_10_Task2_Future_Forecast.png"
plt.savefig(figure_3, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", figure_3)


## 6. Holiday Sales Prediction — Own Model vs AI-Assisted Benchmark


A separate transparent holiday model is used at the **total weekly sales** level so the four named holidays can be compared consistently.

- **Own holiday model:** 90% of the same holiday-week sales from 52 weeks earlier + 10% of the preceding 13-week average sales level.
- **AI-assisted benchmark:** pure 52-week seasonal-naive prediction.

The evaluated holiday weeks are the latest observed occurrences available for all four named holidays within the historical dataset: Thanksgiving 2011, Christmas 2011, Super Bowl 2012 and Labour Day 2012. Each forecast uses only values that would have been known before that target week.


In [ ]:
weekly_total = (
    analysis_data
    .groupby("Date", as_index=False)
    .agg(Actual_Sales=("Weekly_Sales", "sum"))
    .sort_values("Date")
)

weekly_total["Lag_52_Total"] = weekly_total["Actual_Sales"].shift(52)
weekly_total["Recent_13_Week_Mean"] = (
    weekly_total["Actual_Sales"].shift(1).rolling(13).mean()
)

holiday_targets = pd.DataFrame({
    "Date": pd.to_datetime([
        "2011-11-25",
        "2011-12-30",
        "2012-02-10",
        "2012-09-07",
    ]),
    "Holiday_Name": [
        "Thanksgiving",
        "Christmas",
        "Super Bowl",
        "Labour Day",
    ]
})

holiday_predictions = holiday_targets.merge(
    weekly_total,
    on="Date",
    how="left"
)

holiday_predictions["Own_Holiday_Model"] = (
    0.90 * holiday_predictions["Lag_52_Total"]
    + 0.10 * holiday_predictions["Recent_13_Week_Mean"]
)

holiday_predictions["AI_Holiday_Benchmark"] = (
    holiday_predictions["Lag_52_Total"]
)

holiday_predictions.to_csv(
    output_folder / "task2_holiday_predictions.csv",
    index=False
)

display(holiday_predictions.round(2))


In [ ]:
holiday_own_metrics = regression_metrics(
    holiday_predictions["Actual_Sales"],
    holiday_predictions["Own_Holiday_Model"]
)

holiday_ai_metrics = regression_metrics(
    holiday_predictions["Actual_Sales"],
    holiday_predictions["AI_Holiday_Benchmark"]
)

holiday_metrics = pd.DataFrame([
    {"Model": "Own 90% seasonal + 10% recent-level model", **holiday_own_metrics},
    {"Model": "AI-assisted 52-week seasonal naive", **holiday_ai_metrics},
])

holiday_metrics.to_csv(
    output_folder / "task2_holiday_model_metrics.csv",
    index=False
)

display(holiday_metrics.round(3))


In [ ]:
holiday_plot = holiday_predictions.copy()
x = np.arange(len(holiday_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(x - width, holiday_plot["Actual_Sales"] / 1_000_000, width, label="Actual")
ax.bar(x, holiday_plot["Own_Holiday_Model"] / 1_000_000, width, label="Own model")
ax.bar(x + width, holiday_plot["AI_Holiday_Benchmark"] / 1_000_000, width, label="AI benchmark")
ax.set_xticks(x)
ax.set_xticklabels(holiday_plot["Holiday_Name"])
ax.set_ylabel("Total Weekly Sales ($ Millions)")
ax.set_title("Holiday Sales Prediction Comparison")
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.tight_layout()

figure_4 = chart_folder / "Figure_11_Task2_Holiday_Prediction.png"
plt.savefig(figure_4, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", figure_4)


## 7. Task 2 Interpretation Summary


In [ ]:
forecast_winner = model_metrics.sort_values("RMSE").iloc[0]
holiday_rmse_winner = holiday_metrics.sort_values("RMSE").iloc[0]
holiday_wape_winner = holiday_metrics.sort_values("WAPE_Percent").iloc[0]

overall_markdown = markdown_summary[markdown_summary["Type"] == "All"].iloc[0]

print("SALES FORECASTING")
print(
    f"Own model RMSE: ${own_metrics['RMSE']:,.2f}; "
    f"AI benchmark RMSE: ${ai_metrics['RMSE']:,.2f}."
)
print(
    f"Own model WAPE: {own_metrics['WAPE_Percent']:.2f}%; "
    f"AI benchmark WAPE: {ai_metrics['WAPE_Percent']:.2f}%."
)
print("Lower RMSE model:", forecast_winner["Model"])

print("\nMARKDOWN SCENARIO")
print(
    f"Mean model-estimated markdown difference per department-week: "
    f"${overall_markdown['Mean_Impact_Per_Record']:,.2f}."
)
print(
    "This is a model-based association and should not be interpreted as a causal effect."
)

print("\nHOLIDAY PREDICTION")
print(
    f"Own holiday RMSE: ${holiday_own_metrics['RMSE']:,.2f}; "
    f"AI holiday RMSE: ${holiday_ai_metrics['RMSE']:,.2f}."
)
print(
    f"Own holiday WAPE: {holiday_own_metrics['WAPE_Percent']:.2f}%; "
    f"AI holiday WAPE: {holiday_ai_metrics['WAPE_Percent']:.2f}%."
)
print("Lower holiday RMSE:", holiday_rmse_winner["Model"])
print("Lower holiday WAPE:", holiday_wape_winner["Model"])

print("\nFUTURE FORECAST")
print(
    f"Forecast produced for {future_forecast[['Store','Dept']].drop_duplicates().shape[0]} "
    f"active store-department series across {future_forecast['Date'].nunique()} weeks."
)


### Interpretation for the report

The chronological evaluation should be emphasised because it tests the models on later weeks rather than randomly mixed observations. The own Histogram Gradient Boosting lag model should be discussed against the simpler AI-assisted seasonal-naive result using the reported metrics, not only visual similarity.

For holiday prediction, do not force a single "winner" if the metrics disagree. RMSE gives more weight to larger errors, while WAPE measures total absolute forecast error relative to total sales. The comparison is stronger when both are reported and interpreted.

The markdown scenario is exploratory. It can show whether the fitted model associates markdown values with higher or lower sales while holding the other model inputs constant, but it cannot establish causal impact because markdown campaigns may coincide with holidays, promotions or other factors.


## 8. AI-Assisted Comparison and Disclosure Notes


**AI tool:** ChatGPT.

**Task 2 AI use in this contribution:** AI was used to provide a comparative forecasting benchmark and to help explain how the benchmark could be checked against the primary Python model. The numerical benchmark itself is generated reproducibly in Python from the historical data.

**Verification performed:** all AI-assisted comparison values are recalculated from the prepared dataset, compared with the primary model using the same test periods, and evaluated using the same metrics.

**Student/group responsibility:** the group must run and verify the notebook, understand the modelling choices, decide whether they are appropriate for the project, and take responsibility for the final analysis, code, findings and conclusions.

### Example-prompt placeholders for Appendix C

The assessment requires at least three example prompts. **Replace the wording below with the exact prompts from your real ChatGPT history before submission.** Suitable Task 2 prompt types are:

1. Ask ChatGPT for a simple, independently verifiable baseline for weekly retail sales with strong yearly seasonality.
2. Ask ChatGPT how to compare an own forecasting model with an AI-assisted benchmark using RMSE, MAE and WAPE.
3. Ask ChatGPT for a simple holiday-sales benchmark for Super Bowl, Labour Day, Thanksgiving and Christmas, then verify the answer using the group's historical data.

Do not claim prompts were used if they were not actually used.


## 9. Files Created by This Notebook


In [ ]:
print("CSV outputs:")
for file in sorted(output_folder.glob("*.csv")):
    print("-", file.name)

print("\nFigures:")
for file in sorted(chart_folder.glob("*.png")):
    print("-", file.name)
